In [1]:
import torch
import time
from baseformer.scripts.utils import load_model
from baseformer.optim.adamw import AdamW as MyAdamW

DEVICE = "cuda"
MODEL_SIZE = "small"
BATCH_SIZE = 8
SEQ_LEN = 256
WARMUP = 5
STEPS = 20

In [2]:
def benchmark_optimizer(model, optimizer, input_ids, warmup=WARMUP, steps=STEPS):
    """Run forward+backward+step and time just the optimizer step."""
    # Warmup
    for _ in range(warmup):
        loss = model(input_ids).sum()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    torch.cuda.synchronize()
    
    # Timed runs
    times = []
    for _ in range(steps):
        loss = model(input_ids).sum()
        loss.backward()
        torch.cuda.synchronize()
        
        start = time.perf_counter()
        optimizer.step()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
        
        optimizer.zero_grad()
    
    return times

In [3]:
# Benchmark my AdamW
model = load_model(MODEL_SIZE, DEVICE)
input_ids = torch.randint(0, 10048, (BATCH_SIZE, SEQ_LEN), device=DEVICE)

my_optimizer = MyAdamW(model.parameters(), lr=1e-4)
my_times = benchmark_optimizer(model, my_optimizer, input_ids)

del model, my_optimizer
torch.cuda.empty_cache()

In [4]:
# Benchmark torch AdamW
model = load_model(MODEL_SIZE, DEVICE)

torch_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
torch_times = benchmark_optimizer(model, torch_optimizer, input_ids)

del model, torch_optimizer
torch.cuda.empty_cache()

In [ ]:
my_avg = sum(my_times) / len(my_times) * 1000
torch_avg = sum(torch_times) / len(torch_times) * 1000

print(f"My AdamW:    {my_avg:.2f} ms")
print(f"Torch AdamW: {torch_avg:.2f} ms")
print(f"Ratio:       {my_avg / torch_avg:.2f}x")

My AdamW:    4.05 ms
Torch AdamW: 4.08 ms
Ratio:       0.99x


: 